# List COGs for collection `2024_bci`

This notebook queries the Kanopia STAC API and lists all COG assets matching the `2024_bci` collection and the `*bciwhole*rgb.cog.tif` pattern. It is based on the structure of `workshop_kanopia_stac_cog_python.ipynb`.

In [21]:
# STAC API URL (no Basic Auth needed)
STAC_API_URL = "https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac/"
COLLECTION_ID = "2024_bci"

print("STAC endpoint:", STAC_API_URL)
print("Collection:", COLLECTION_ID)

STAC endpoint: https://kanopia.org/stac-fastapi-pgstac/api/v1/pgstac/
Collection: 2024_bci


In [22]:
import sys
import subprocess

# List your packages directly
packages = ["requests", "rasterio", "pystac_client", "geopandas", "imageio"]

print(f"Installing: {packages}")
subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

print("Done. If imports fail, go to 'Runtime' -> 'Restart session'.")


Installing: ['requests', 'rasterio', 'pystac_client', 'geopandas', 'imageio']
Defaulting to user installation because normal site-packages is not writeable
Done. If imports fail, go to 'Runtime' -> 'Restart session'.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [23]:
# Optional dependencies: pystac-client
# Install in your environment if missing:
#   python -m pip install pystac-client

from pystac_client import Client
import re

pattern = re.compile(r"(\d{8})_bciwhole_.*rgb\.cog\.tif")

client = Client.open(STAC_API_URL)
search = client.search(collections=[COLLECTION_ID], max_items=500)

# Support both older and new .get_all_items behaviour
items = search.get_all_items()
if hasattr(items, 'features'):
    items = items.features

cog_list = []
for item in items:
    assets = item.assets if hasattr(item, 'assets') else item.get('assets', {})
    item_id = getattr(item, 'id', None) or item.get('id', None)

    for key, asset in (assets.items() if isinstance(assets, dict) else []):
        if not asset:
            continue
        href = asset.href if hasattr(asset, 'href') else asset.get('href', '')
        if not href:
            continue

        if pattern.search(href):
            date_int = int(pattern.search(href).group(1))
            cog_list.append({
                'item_id': item_id,
                'asset_key': key,
                'href': href,
                'date': date_int,
            })

# Sort by date
cog_list.sort(key=lambda x: x['date'])

print(f"Found {len(cog_list)} matching COG assets for collection {COLLECTION_ID}")
for i, c in enumerate(cog_list, start=1):
    print(f"{i:03d}: {c['date']} | {c['item_id']} | {c['asset_key']} | {c['href']}")

Found 17 matching COG assets for collection 2024_bci
001: 20240611 | 20240611_bciwhole_rx1rii | rgb/optimized | https://lab.kanopia.org/share/1/20240611_bciwhole_rx1rii/20240611_bciwhole_rx1rii_rgb.cog.tif
002: 20240716 | 20240716_bciwhole_rx1rii | rgb/optimized | https://lab.kanopia.org/share/1/20240716_bciwhole_rx1rii/20240716_bciwhole_rx1rii_rgb.cog.tif
003: 20240813 | 20240813_bciwhole_rx1rii | rgb/optimized | https://lab.kanopia.org/share/1/20240813_bciwhole_rx1rii/20240813_bciwhole_rx1rii_rgb.cog.tif
004: 20240918 | 20240918_bciwhole_rx1rii | rgb/optimized | https://lab.kanopia.org/share/1/20240918_bciwhole_rx1rii/20240918_bciwhole_rx1rii_rgb.cog.tif
005: 20241014 | 20241014_bciwhole_rx1rii | rgb/optimized | https://lab.kanopia.org/share/1/20241014_bciwhole_rx1rii/20241014_bciwhole_rx1rii_rgb.cog.tif
006: 20241112 | 20241112_bciwhole_rx1rii | rgb/optimized | https://lab.kanopia.org/share/1/20241112_bciwhole_rx1rii/20241112_bciwhole_rx1rii_rgb.cog.tif
007: 20241216 | 20241216_bciw

In [ ]:
# --- New workflow: extract 512x512 PNG centered on a point in a reference geopackage ---
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.windows import Window
from pyproj import Transformer

# Credentials for COG access
COG_USER = "panama"
COG_PASS = "panama123"

def add_basic_auth_to_url(url, user, pwd):
    from urllib.parse import urlparse, urlunparse
    parsed = urlparse(url)
    if parsed.username is not None:
        # Already has credentials
        return url
    netloc = f"{user}:{pwd}@" + parsed.netloc
    return urlunparse((parsed.scheme, netloc, parsed.path, parsed.params, parsed.query, parsed.fragment))

# Adjust path if needed
GPKG_PATH = "https://lab.kanopia.org/share/public/2024-05-14_2024-06-11_fieldDatasheet.gpkg"
TARGET_FIELDS = {
    "treetag": "6949",
    "polygon_id": 1041,
    "fid": 11,
}
OUTPUT_DIR = Path("bci_cog_snips")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading geopackage:", GPKG_PATH)

# load all layers and find first layer with matching rows
gpkg = gpd.read_file(GPKG_PATH)
print("Geopackage layer rows:", len(gpkg))

cond = (
    (gpkg["TreeTag"].astype(str) == str(TARGET_FIELDS["treetag"])) &
    (gpkg["Polygon_ID"] == TARGET_FIELDS["polygon_id"])
)

matches = gpkg.loc[cond]
if matches.empty:
    raise ValueError("No matching row found for treetag/polygon_id/fid in geopackage")

# Use the first geometry in case of duplicates
geom = matches.geometry.iloc[0]
if geom is None:
    raise ValueError("Geometry is empty for selected record")

point = geom.centroid if not geom.geom_type == "Point" else geom
x, y = point.x, point.y
n = len(cog_list)
print(f"Found point at {x:.6f}, {y:.6f} (from {len(matches)} matching records). Processing {n} COGs...")

for idx, c in enumerate(cog_list, start=1):
    href = c["href"]
    # Add Basic Auth to COG URL
    href_auth = add_basic_auth_to_url(href, COG_USER, COG_PASS)
    name = f"{c['asset_key']}"
    out_png = OUTPUT_DIR / f"{name}.png"
    out_png.parent.mkdir(parents=True, exist_ok=True)  # Ensure directory exists

    with rasterio.open(f"/vsicurl/{href_auth}") as src:
        # Reproject point if CRS differ
        if gpkg.crs != src.crs:
            transformer = Transformer.from_crs(gpkg.crs, src.crs, always_xy=True)
            x_proj, y_proj = transformer.transform(x, y)
        else:
            x_proj, y_proj = x, y

        # convert world coordinate to image row/col
        col, row = src.index(x_proj, y_proj)

        half = 256
        w = Window(col - half, row - half, 512, 512)
        w = w.intersection(Window(0, 0, src.width, src.height))

        if w.width < 1 or w.height < 1:
            print(f"[{idx}/{n}] skipped (window out of bounds): {name}")
            continue

        # read up to 3 bands for RGB-like output
        band_count = min(src.count, 3)
        arr = src.read(list(range(1, band_count + 1)), window=w)

        if band_count == 1:
            img = np.squeeze(arr)
            img = np.stack([img, img, img], axis=-1)
        else:
            img = np.transpose(arr, (1, 2, 0))

        # scale to 0..255 for PNG
        img_min, img_max = np.nanmin(img), np.nanmax(img)
        if img_max > img_min:
            img = (img - img_min) / (img_max - img_min) * 255
        img = np.nan_to_num(img, nan=0.0, posinf=255.0, neginf=0.0)
        img = np.clip(img, 0, 255).astype(np.uint8)

        # write PNG
        from imageio import imwrite

        imwrite(str(out_png), img)
        print(f"[{idx}/{n}] wrote {out_png}")

print("Done: extracted PNGs in", OUTPUT_DIR)

In [30]:
print(f"TreeTag {TARGET_FIELDS['treetag']} point coordinates: lat={y:.6f}, lon={x:.6f}")

TreeTag 6949 point coordinates: lat=1011873.444245, lon=625956.044784


In [26]:
# Print CRS of geopackage and first COG before processing
print('Geopackage CRS:', gpkg.crs)
if len(cog_list) > 0:
    test_href = add_basic_auth_to_url(cog_list[0]['href'], COG_USER, COG_PASS)
    with rasterio.open(f'/vsicurl/{test_href}') as src:
        print('First COG CRS:', src.crs)

Geopackage CRS: EPSG:32617
First COG CRS: EPSG:32617


In [12]:
# Print all column names and first few rows to inspect attribute names
print("Geopackage columns:", gpkg.columns.tolist())
print(gpkg.head())

Geopackage columns: ['Species', 'POM_m', 'Buttresses_count', 'Status', 'Mode', 'Crown_intactness', 'Crow_loss', 'Liana_observed', 'Liana_pre_damage', 'Leaf_presence', 'Biotic_damage', 'Heart_rot', 'Notes', 'Alive flag', 'Mode flag', 'Photos', 'TreeDate', 'Biotic damage length', 'Alive flag extra', 'Mode flag extra', 'Butresses_height_m', 'Type', 'longitude', 'latitude', 'Angle', 'Distance_m', 'DBH_cm', 'Height_of_break_m', 'Polygon_ID', 'TreeTag', 'geometry']
  Species  POM_m  Buttresses_count Status Mode Crown_intactness Crow_loss  \
0     NaN   6.63                 3      0    U               40        60   
1  STERPE   2.85                 2      0    S              100         0   
2   VIRSE   1.30                 0      0    S               60        40   
3   PRICO   1.30                 0      0    S              100         0   
4   TACVE   4.40                 4      0    U               80        20   

  Liana_observed Liana_pre_damage Leaf_presence  ... Type      longitude 

## Notes
- If no entries are found, ensure the `COLLECTION_ID` exists and the API endpoint is available.
- You can customize `max_items`, regex pattern, or add grid mapping to focus on date areas.